In [ ]:
import sys

sys.path.insert(0, "/Users/giladrubin/python_workspace/hypercache/src")
sys.path.insert(0, "/Users/giladrubin/python_workspace/hypergraph/src")

print("hypercache ok")
print("hypergraph ok")

In [ ]:
# -----------------------------------------------------------
# 1. Define a component with cached methods
# -----------------------------------------------------------
from datetime import timedelta

from hypercache import CachePolicy, CacheService, cached


class Embedder:
    """Wraps an embedding model. Caches results by text + model."""

    def __init__(self, cache: CacheService) -> None:
        self._cache = cache
        self.model = "text-embedding-3-small"
        self.call_count = 0

    @cached(
        version="embed:v1",
        policy=CachePolicy(ttl=timedelta(hours=6)),
    )
    def embed(self, text: str) -> list[float]:
        """Return a fake embedding vector (counts real calls)."""
        self.call_count += 1
        return [0.1, 0.2, 0.3, 0.4]


print("Embedder defined")

In [ ]:
# -----------------------------------------------------------
# 2. Define a graph that uses the component
# -----------------------------------------------------------
from hypergraph import Graph, node

cache = CacheService.memory()
embedder = Embedder(cache)


@node(output_name="embedding")
def embed(text: str) -> list[float]:
    return embedder.embed(text).value


@node(output_name="answer")
def generate(embedding: list, text: str) -> str:
    return f"Answer to {text!r} (embedding dim={len(embedding)})"


graph = Graph([embed, generate], name="rag")
print("Graph ready:", graph)

In [ ]:
# -----------------------------------------------------------
# 3. Observe component cache events via TypedEventProcessor
#    Zero boilerplate on the component or node side.
# -----------------------------------------------------------
from hypergraph.events.processor import TypedEventProcessor
from hypergraph.events.types import ComponentCacheEvent
from hypergraph.runners.sync import SyncRunner


class CacheLogger(TypedEventProcessor):
    """Prints a line for every component-level cache decision."""

    def on_component_cache(self, event: ComponentCacheEvent) -> None:
        icon = "↩ hit" if event.hit else "· miss"
        stale = " (stale)" if event.stale else ""
        refresh = " [refresh started]" if event.refreshing else ""
        print(f"  {icon}{stale}{refresh}  {event.component}.{event.operation}")


runner = SyncRunner()
print("Runner and processor ready")

In [ ]:
# -----------------------------------------------------------
# 4. Run 1: cold start — everything is a miss
# -----------------------------------------------------------
print("=== Run 1: cold start ===")
result = runner.run(
    graph,
    text="what is machine learning?",
    event_processors=[CacheLogger()],
)
print(f"answer: {result['answer']}")
print(f"real embed() calls: {embedder.call_count}")

In [ ]:
# -----------------------------------------------------------
# 5. Run 2: same input — embed() served from cache
# -----------------------------------------------------------
print("=== Run 2: same input ===")
result = runner.run(
    graph,
    text="what is machine learning?",
    event_processors=[CacheLogger()],
)
print(f"answer: {result['answer']}")
print(f"real embed() calls still: {embedder.call_count}  ← no new call")

In [ ]:
# -----------------------------------------------------------
# 6. Collect events as structured data (no printing needed)
# -----------------------------------------------------------


class CacheRecorder(TypedEventProcessor):
    def __init__(self):
        self.events: list[ComponentCacheEvent] = []

    def on_component_cache(self, event: ComponentCacheEvent) -> None:
        self.events.append(event)


recorder = CacheRecorder()
runner.run(graph, text="what is deep learning?", event_processors=[recorder])
runner.run(graph, text="what is deep learning?", event_processors=[recorder])
runner.run(graph, text="what is ML?", event_processors=[recorder])

for e in recorder.events:
    print(f"node={e.node_name!r:12}  hit={e.hit!s:5}  wrote={e.wrote!s:5}  op={e.operation!r}")

In [ ]:
# -----------------------------------------------------------
# 7. Stale window + background refresh demo
# -----------------------------------------------------------
import time
from datetime import timedelta

from hypercache import CachePolicy, CacheService, cached


class SlowEmbedder:
    def __init__(self, cache):
        self._cache = cache
        self.call_count = 0

    @cached(
        version="v1",
        policy=CachePolicy(
            ttl=timedelta(seconds=10),
            stale=timedelta(milliseconds=100),  # goes stale after 100ms
            refresh_in_background=True,
        ),
    )
    def embed(self, text: str) -> list[float]:
        self.call_count += 1
        return [float(self.call_count)]


cache2 = CacheService.memory()
se = SlowEmbedder(cache2)


@node(output_name="embedding")
def slow_embed(text: str) -> list[float]:
    return se.embed(text).value


graph2 = Graph([slow_embed], name="stale-demo")
runner2 = SyncRunner()

recorder2 = CacheRecorder()

print("Run 1: cold miss")
runner2.run(graph2, text="hello", event_processors=[recorder2])

print("Run 2: fresh hit (within stale window)")
runner2.run(graph2, text="hello", event_processors=[recorder2])

time.sleep(0.15)  # let it go stale

print("Run 3: stale hit + background refresh triggered")
runner2.run(graph2, text="hello", event_processors=[recorder2])

for e in recorder2.events:
    status = "hit" if e.hit else "miss"
    stale = "+stale" if e.stale else ""
    refresh = "+refreshing" if e.refreshing else ""
    print(f"  {status}{stale}{refresh}  wrote={e.wrote}")

In [ ]:
# -----------------------------------------------------------
# 8. Progress bar — ↩ and ↻ symbols appear live in TTY
#    (forced non-tty here, so shows text fallback)
# -----------------------------------------------------------

# In a real terminal this renders a Rich progress bar with:
#   embed   ━━━━━ 1/1  0:00:00  ↩1
#   generate ━━━━━ 1/1  0:00:00
#
# ↩ = component cache hits inside that node
# ↻ = background stale refreshes triggered

runner.run(
    graph,
    text="what is machine learning?",
    show_progress=True,  # auto-adds RichProgressProcessor
)